# 4.4 讀完前文後，每位置自己的特徵怎樣加工？


注意力讓當前位置從其他位置帶回資料；接著還需要把手上的一排特徵組合成新特徵。這個工作在各位置獨立完成，叫**前饋網路**（Feed-Forward Network，FFN）。

本節用同一套配方處理兩個位置，各輸入 `[1,1,1,1]`。先將四格擴大為十六個中間特徵，經過非線性轉換，再縮回四格。擴張讓同一排輸入先產生更多不同的特徵組合，各自經非線性加工，再由縮回層混成四格修正。擴張的是特徵，不是文字位置；縮回則讓結果能加到四格的殘差主路。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.modern import DenseFFN

torch.manual_seed(42)
layer = DenseFFN(4)
x = torch.ones(1, 2, 4)
y = layer(x)
print("擴張權重", layer.up.weight.shape)
print("縮回權重", layer.down.weight.shape)
print(y.shape, y)
assert torch.allclose(y[:, 0], y[:, 1])

擴張權重 `[16,4]`，縮回權重 `[4,16]`，對應四到十六、十六到四兩套 Linear。非線性叫**高斯誤差線性單元**（Gaussian Error Linear Unit，GELU），會平滑地調整每個數的大小，較正的通常保留更多、較負的被縮小，也可能保留小負值；它不同於 ReLU 直接清掉所有負數。本例選 GELU 來示範平滑調整的非線性，四倍寬度與這個選擇是所用工具的設定；這次只檢查計算，不比較它們與其他設定的效果。

輸出仍 `[1,2,4]`。兩個位置輸入完全相同，又使用同一配方，因此輸出兩排逐格相同，斷言核對的是值，不只是形狀。權重從亂數開始，這些輸出沒有固定字義。

這裡的 **Dense** 表示每個位置都用完整這套 FFN，不依當前字只選部分配方。共享配方不等於位置互相讀取：改變位置 1，只會透過此 FFN 改位置 1，跨位置交流由注意力做。

練習儲存 y 後，改 `x[0,1,0]=2`，再算 `changed=layer(x)`。第 0 位置應與原 y 相同，第 1 位置通常不同。用逐格比較核對，分清「每位置內部加工」和「跨位置讀取」。

<details>
<summary>補充與重做</summary>

Linear 見 [2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)，非線性為何有用見 [2.4](https://birdhackor.github.io/tiny-perceptron-vlm/2.4.html)。`manual_seed(42)` 固定初始化起點，這次沒有答案、求導或更新。GELU 的完整公式暫不需要；本節關心非線性放在擴張與縮回之間，以及位置數不變。

</details>
